# Part 4: the operator window on the live Earth Engine map

This notebook is a small version of the EarthQuery operator tool, with the same three panels and the same names. The model is trained inside Google Earth Engine on the AlphaEarth embeddings under your labelled points. Earth Engine then scores every 10 m pixel of Austria and draws the result as map tiles, so nothing is downloaded and the map updates a few seconds after you retrain. The search for candidates works on a pyramid of the score map:

![the search ladder](../figures/search_ladder.png)

You need an Earth Engine account and a Cloud project (see the README). Put the project id below. The first time, a link appears: open it, allow access and paste the code back into the notebook.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # makes `al_training` importable from this folder
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
os.makedirs("../outputs", exist_ok=True)

In [ ]:
from al_training.earth_engine import init_ee, alphaearth, austria, load_seeds, pool_farms, points_layer, Session, AEE_VIS

EE_PROJECT = os.environ.get("EE_PROJECT", "nmap-454717")     # <- put your project id here
init_ee(EE_PROJECT)

## The AlphaEarth layer: the three most important bands as red, green and blue

AlphaEarth gives every 10 m pixel 64 numbers. Here the three that matter most to the Austrian solar farm model are shown as red, green and blue: A22, A01 and A19, the dimensions with the largest mean SHAP value in the random forest of the three-country study. Solar farms sit low on A22 and A01 and high on A19, so they come out dark blue; similar colours are similar places. The model works on all 64 numbers. The orange points are the 259 solar farms of the pool of part 2; zoom in on one to compare its colour with its surroundings.

In [ ]:
import geemap

farms = pool_farms()                                     # the 259 solar farms of the pool of part 2
m = geemap.Map(center=(47.6, 14.2), zoom=7, basemap="Esri.WorldImagery", height="480px")
m.addLayer(alphaearth().clip(austria()), AEE_VIS, "AlphaEarth, top 3 SHAP bands A22 A01 A19 as RGB")
m.addLayer(points_layer(farms), {}, f"solar farms of the pool ({len(farms)})")
m

## The operator window: Classification, Similarity Search and Active Learning

The session starts with a few labels: 10 solar farms from the Microsoft layer and 60 locations an operator marked as not a farm. Run the cell. The map is on the left; on the right are the three panels of the tool:

* **Classification**: **Add positive at centre** and **Add negative at centre** label the point in the middle of the map (the tool's + and - keys). Clicking on the map also adds a point; the dropdown says which kind. **Undo last** removes the last point, **Save as CSV** writes them all.
* **Similarity Search**: choose a method and press **Run Similarity**. **Cosine Similarity** needs positives only and scores every pixel by how much its embedding points in the direction of the mean solar farm; this is the cold-start scorer of the tool. **Random Forest** needs positives and negatives and gives the probability of a solar farm. Either way Earth Engine draws the score for the whole country (dark = low, yellow = high). The layer control at the top right switches layers on and off.
* **Active Learning**: **Find Top Locations** runs the three-criteria rule of part 1 on the score map and lists the candidates. **Prev** and **Next** move the map to them.
* **Basemap** switches between Esri World Imagery and Google imagery (satellite, or hybrid with labels). The two are often from different dates; the tool's jury compares both before it writes a label.

The status line under the panels says what is happening; every action reports at once, and Earth Engine needs 5 to 15 seconds to train and a minute to search all of Austria.

In [ ]:
session = Session(labels=load_seeds(n_pos=10, n_neg=60))
session.show()

In [ ]:
session.run_similarity()          # the same as pressing Run Similarity (random forest by default)

## Adding labels at the map centre

Zoom in somewhere, put a solar farm or a field in the middle of the map, and press **Add positive at centre** or **Add negative at centre**. Then **Run Similarity** again: the score map changes around the new label. Every point is one row in `session.labels`.

In [ ]:
session.labels.tail()

## Find Top Locations: the three-criteria rule on the score map

Pressing **Find Top Locations** (or running the cell below) runs the EarthQuery rule on the live score map:

1. Earth Engine finds the hot spots of the score map at the discovery scale (320 m by default) over the search area, traces them into blobs and returns the 150 best with their peak score and location.
2. The batch is filled in three parts: exploit (highest scores), diversity (one per embedding cluster) and novelty (high score, far in embedding space from anything labelled). Picks keep the minimum distance from each other.
3. Each winner is moved to the best 10 m pixel near it.

Over all of Austria this takes about a minute. The candidates appear as numbered markers coloured by the criterion that chose them (gold exploit, teal diversity, red novelty), and the list in the panel shows their scores. Press **Next** to fly to each one, look at the imagery, and answer with **Add positive at centre** or **Add negative at centre**: the answer is recorded against the candidate. Then **Run Similarity** again.

In [ ]:
candidates = session.find_top(n=12)
candidates

In [ ]:
session.run_similarity()          # after answering: retrain and see the map change

## Searching only the current map view

For a quick loop, set the search area to the current map view (or pass it in code). The view is small, so the finer 160 m discovery scale is fast enough here.

In [ ]:
session.find_top(n=6, region=session.view_region(), scale=160)

## What the full EarthQuery tool adds to this notebook

The full EarthQuery tool runs this same loop with a few more parts:

* **A ladder of scales.** A campaign region is searched in seven passes, three at 2,560 m with 60 candidates and 5 km spacing, two at 160 m with 120 candidates, two at 20 m with 240 candidates, and the model is retrained after every pass. Coarse passes find the areas, fine passes locate the farms. Searches at 40 m and finer run as background exports.
* **Thresholds by percentile, never by value.** Scores are not comparable between regions (European negatives can score higher than African farms), so every cut-off is a percentile of the region's own score map.
* **No repeats.** Every answered or pending point is masked with a buffer before the next search, and picks keep a minimum distance from each other.
* **Seeds.** A region starts from the verified Microsoft sites (at most 400, condensed with TypiClust), borrowed neighbours when there are fewer than 100, and 200 landscape negatives found with k-means over a 5 km grid.
* **Many annotators and a jury.** People label on desktop or phone with a job queue, or two vision language models vote on two basemaps and a person only breaks the ties. Every answer carries its round, session and imagery date.
* **Cost.** One full ladder over a region uses about 0.11% of the monthly Earth Engine quota; the jury's inference, not the search, is the bottleneck.

## Saving the labels to CSV

In [ ]:
session.save("../outputs/my_austria_labels.csv")
print(f"{len(session.labels)} labels, {len(session.rounds)} training rounds")

## Exercise: start with no labels and the cosine scorer

Start a session without any labels. The map opens on a solar farm near Neusiedl am See. Press **Add positive at centre**, choose **Cosine Similarity** and press **Run Similarity**: one positive is enough for this scorer. Look at the score map, press **Find Top Locations**, and answer the candidates. Once you have a few negatives, switch to **Random Forest**. How many answers does it take before the score map is high only on solar farms?

In [ ]:
cold = Session(labels=None, center=(47.9697, 16.3372), zoom=16, method="cosine")
cold.show()

## Summary

* The whole loop of parts 1 and 2 runs here on the real data: your answers are the labels, Earth Engine is the model and the scoring, and the score map is the model's prediction.
* Find Top Locations implements the strategy: exploit finds farms, diversity and novelty keep the batch from concentrating on one type of place.
* The EarthQuery tool adds what a notebook cannot: many annotators at once, a database of every answer, a ladder of search scales, and a jury of vision language models when no person is available.